In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

file_path = "/content/final_regression_dataset_v2.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="regression_base")

base_2022 = df[df["fiscal_year"] == 2022].copy()

print(base_2022.shape)
base_2022[["firm_id", "book_leverage", "interest_coverage"]].head()

(196, 30)


,firm_id,book_leverage,interest_coverage
0,F001,0.317002,40.749574
4,F002,0.152152,20.439599
8,F003,0.247720,42.546610
12,F004,0.145134,5.174482
16,F005,0.040248,209.641457


In [3]:
lev_cut = base_2022["book_leverage"].quantile(0.67)
ic_cut = base_2022["interest_coverage"].quantile(0.33)

print("Leverage cutoff (67th percentile):", lev_cut)
print("Interest coverage cutoff (33rd percentile):", ic_cut)

Leverage cutoff (67th percentile): 0.3167675675576231
Interest coverage cutoff (33rd percentile): 9.57081452164009


In [4]:
base_2022["vulnerable_2022"] = (
    (base_2022["book_leverage"] >= lev_cut) &
    (base_2022["interest_coverage"] <= ic_cut)
).astype(int)

print(base_2022["vulnerable_2022"].value_counts())

vulnerable_2022
0    162
1     34
Name: count, dtype: int64


In [5]:
vuln_map = base_2022[["firm_id", "vulnerable_2022"]].copy()

df = df.merge(vuln_map, on="firm_id", how="left")

print(df["vulnerable_2022"].value_counts(dropna=False))

vulnerable_2022
0.0    639
1.0    136
NaN      6
Name: count, dtype: int64


In [6]:
df["rate_change_x_vulnerable_2022"] = df["policy_rate_change"] * df["vulnerable_2022"]

df[[
    "firm_id",
    "fiscal_year",
    "policy_rate_change",
    "vulnerable_2022",
    "rate_change_x_vulnerable_2022"
]].head()

,firm_id,fiscal_year,policy_rate_change,vulnerable_2022,rate_change_x_vulnerable_2022
0,F001,2022,NaN,0.0,NaN
1,F001,2023,1.23,0.0,0.0
2,F001,2024,-0.85,0.0,-0.0
3,F001,2025,-0.76,0.0,-0.0
4,F002,2022,NaN,0.0,NaN


In [7]:
reg_cols = [
    "firm_id",
    "fiscal_year",
    "roa_ebit",
    "policy_rate_change",
    "vulnerable_2022",
    "rate_change_x_vulnerable_2022",
    "size_ln_assets"
]

reg_vuln = df[reg_cols].dropna().copy()

print("Observations:", len(reg_vuln))
print("Firmes:", reg_vuln["firm_id"].nunique())
print("Années:", sorted(reg_vuln["fiscal_year"].unique()))

Observations: 576
Firmes: 195
Années: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [8]:
model_vuln_roaebit = smf.ols(
    "roa_ebit ~ policy_rate_change + rate_change_x_vulnerable_2022 + size_ln_assets + C(firm_id) + C(fiscal_year)",
    data=reg_vuln
).fit(cov_type="cluster", cov_kwds={"groups": reg_vuln["firm_id"]})

print(model_vuln_roaebit.summary())

                            OLS Regression Results                            
Dep. Variable:               roa_ebit   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     9796.
Date:                Sun, 05 Apr 2026   Prob (F-statistic):          4.82e-231
Time:                        23:00:02   Log-Likelihood:                 1306.4
No. Observations:                 576   AIC:                            -2213.
Df Residuals:                     376   BIC:                            -1342.
Df Model:                         199                                         
Covariance Type:              cluster                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 199, but rank is 5
  warnings.warn('covariance of constraints does not have full '


In [9]:
main_vars = [
    "policy_rate_change",
    "rate_change_x_vulnerable_2022",
    "size_ln_assets"
]

results_vuln_roaebit = pd.DataFrame({
    "coef": model_vuln_roaebit.params[main_vars],
    "std_err": model_vuln_roaebit.bse[main_vars],
    "t_stat": model_vuln_roaebit.tvalues[main_vars],
    "p_value": model_vuln_roaebit.pvalues[main_vars]
})

print(results_vuln_roaebit)

print("\nN observations :", int(model_vuln_roaebit.nobs))
print("R² :", round(model_vuln_roaebit.rsquared, 4))
print("Adj. R² :", round(model_vuln_roaebit.rsquared_adj, 4))

print("\nCoefficient clé :")
print("rate_change_x_vulnerable_2022 =", round(model_vuln_roaebit.params["rate_change_x_vulnerable_2022"], 6))
print("p-value =", round(model_vuln_roaebit.pvalues["rate_change_x_vulnerable_2022"], 6))

                                   coef   std_err    t_stat   p_value
policy_rate_change            -0.002170  0.003027 -0.716977  0.473388
rate_change_x_vulnerable_2022  0.005811  0.004059  1.431780  0.152207
size_ln_assets                 0.015143  0.050089  0.302318  0.762410

N observations : 576
R² : 0.8736
Adj. R² : 0.8067

Coefficient clé :
rate_change_x_vulnerable_2022 = 0.005811
p-value = 0.152207


In [10]:
import pandas as pd
import statsmodels.formula.api as smf

file_path = "/content/final_regression_dataset_v2.xlsx"   # adapte si besoin
df = pd.read_excel(file_path, sheet_name="regression_base")

# Base 2022 pour définir la vulnérabilité initiale
base_2022 = df[df["fiscal_year"] == 2022].copy()

# Seuils
lev_cut = base_2022["book_leverage"].quantile(0.67)
ic_cut = base_2022["interest_coverage"].quantile(0.33)

# Dummy de vulnérabilité
base_2022["vulnerable_2022"] = (
    (base_2022["book_leverage"] >= lev_cut) &
    (base_2022["interest_coverage"] <= ic_cut)
).astype(int)

# Merge dans la base panel
vuln_map = base_2022[["firm_id", "vulnerable_2022"]].copy()
df = df.merge(vuln_map, on="firm_id", how="left")

# Interaction avec la variation des taux
df["rate_change_x_vulnerable_2022"] = df["policy_rate_change"] * df["vulnerable_2022"]

print(df[["firm_id", "fiscal_year", "policy_rate_change", "vulnerable_2022", "rate_change_x_vulnerable_2022"]].head())
print(df["vulnerable_2022"].value_counts(dropna=False))

  firm_id  fiscal_year  policy_rate_change  vulnerable_2022  \
0    F001         2022                 NaN              0.0   
1    F001         2023                1.23              0.0   
2    F001         2024               -0.85              0.0   
3    F001         2025               -0.76              0.0   
4    F002         2022                 NaN              0.0   

   rate_change_x_vulnerable_2022  
0                            NaN  
1                            0.0  
2                           -0.0  
3                           -0.0  
4                            NaN  
vulnerable_2022
0.0    639
1.0    136
NaN      6
Name: count, dtype: int64


In [11]:
reg_cols = [
    "firm_id",
    "fiscal_year",
    "interest_coverage",
    "policy_rate_change",
    "vulnerable_2022",
    "rate_change_x_vulnerable_2022",
    "size_ln_assets"
]

reg_last = df[reg_cols].dropna().copy()

print("Observations:", len(reg_last))
print("Firmes:", reg_last["firm_id"].nunique())
print("Années:", sorted(reg_last["fiscal_year"].unique()))

Observations: 573
Firmes: 195
Années: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [12]:
model_last = smf.ols(
    "interest_coverage ~ policy_rate_change + rate_change_x_vulnerable_2022 + size_ln_assets + C(firm_id) + C(fiscal_year)",
    data=reg_last
).fit(cov_type="cluster", cov_kwds={"groups": reg_last["firm_id"]})

print(model_last.summary())

                            OLS Regression Results                            
Dep. Variable:      interest_coverage   R-squared:                       0.703
Model:                            OLS   Adj. R-squared:                  0.545
Method:                 Least Squares   F-statistic:                     170.1
Date:                Sun, 05 Apr 2026   Prob (F-statistic):           6.39e-69
Time:                        23:04:42   Log-Likelihood:                -3097.4
No. Observations:                 573   AIC:                             6595.
Df Residuals:                     373   BIC:                             7465.
Df Model:                         199                                         
Covariance Type:              cluster                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 199, but rank is 5
  warnings.warn('covariance of constraints does not have full '


In [13]:
main_vars = [
    "policy_rate_change",
    "rate_change_x_vulnerable_2022",
    "size_ln_assets"
]

results_last = pd.DataFrame({
    "coef": model_last.params[main_vars],
    "std_err": model_last.bse[main_vars],
    "t_stat": model_last.tvalues[main_vars],
    "p_value": model_last.pvalues[main_vars]
})

print(results_last)

print("\nN observations :", int(model_last.nobs))
print("R² :", round(model_last.rsquared, 4))
print("Adj. R² :", round(model_last.rsquared_adj, 4))

print("\nCoefficient clé :")
print("rate_change_x_vulnerable_2022 =", round(model_last.params["rate_change_x_vulnerable_2022"], 6))
print("p-value =", round(model_last.pvalues["rate_change_x_vulnerable_2022"], 6))

                                    coef    std_err    t_stat   p_value
policy_rate_change            -22.389591  10.830194 -2.067331  0.038703
rate_change_x_vulnerable_2022  -1.019303   3.859655 -0.264092  0.791709
size_ln_assets                 84.716384  66.513023  1.273681  0.202776

N observations : 573
R² : 0.7033
Adj. R² : 0.545

Coefficient clé :
rate_change_x_vulnerable_2022 = -1.019303
p-value = 0.791709
